In [1]:
import json
from typing import Dict, List, Optional
from dataclasses import dataclass
import os
import re
import traceback

# AWS SDK (boto3) 가용성 확인
try:
    import boto3
    BOTO3_AVAILABLE = True
except ImportError:
    BOTO3_AVAILABLE = False
    print("⚠️ boto3가 설치되지 않았습니다. Claude API 호출이 불가능합니다.")


# ============================================================================
# 데이터 클래스 (V5.X 구조 반영)
# ============================================================================

@dataclass
class ClassifiedWord:
    """분류된 단어 (하위 엔터티 타입별)"""
    word: str
    entity_type: str


@dataclass
class KeywordResult:
    """키워드 생성 결과"""
    product_name: str
    type_definitions: Dict[str, str] # LLM이 정의한 엔티티 타입 목록
    combination_patterns: List[str]
    classified_words: List[ClassifiedWord]
    entities_by_type: Dict[str, List[str]] # LLM이 정의한 엔터티 타입별 단어 목록
    keywords_positive: List[str]
    keywords_negative: List[str]
    original_text_preview: str


# ============================================================================
# 키워드 생성기 클래스
# ============================================================================

class KeywordGenerator:
    """
    V5.X 유연 엔티티 형식에 맞춰 Claude 3.5 Sonnet 모델을 사용해 키워드 생성 및 파싱
    """

    def __init__(self, region: str = "ap-northeast-2"):
        self.region = region
        self.total_input_tokens = 0
        self.total_output_tokens = 0

        if BOTO3_AVAILABLE:
            try:
                self.bedrock = boto3.client(
                    service_name='bedrock-runtime',
                    region_name=region,
                    config=boto3.session.Config(
                        read_timeout=300,
                        retries={'max_attempts': 3}
                    )
                )
                print("✅ AWS Bedrock 클라이언트 초기화 완료")
            except Exception as e:
                print(f"⚠️ AWS Bedrock 초기화 실패: {e}")
                self.bedrock = None
        else:
            self.bedrock = None

    def calculate_cost(self, input_tokens: int, output_tokens: int) -> dict:
        """토큰 사용량 기반 비용 계산 (Claude 3.5 Sonnet 가격 기준)"""
        INPUT_PRICE_PER_1M = 3.00
        OUTPUT_PRICE_PER_1M = 15.00
        USD_TO_KRW = 1400

        input_cost_usd = (input_tokens / 1000000) * INPUT_PRICE_PER_1M
        output_cost_usd = (output_tokens / 1000000) * OUTPUT_PRICE_PER_1M
        total_cost_usd = input_cost_usd + output_cost_usd
        total_cost_krw = total_cost_usd * USD_TO_KRW

        return {
            'input_tokens': input_tokens,
            'output_tokens': output_tokens,
            'input_cost_usd': input_cost_usd,
            'output_cost_usd': output_cost_usd,
            'total_cost_usd': total_cost_usd,
            'total_cost_krw': total_cost_krw
        }

    def get_total_cost(self) -> dict:
        return self.calculate_cost(self.total_input_tokens, self.total_output_tokens)

    def _parse_v5_4_output(self, content: str) -> Dict[str, str]:
        """### 구분자 기반 출력을 파싱 (최종 보강)"""
        sections = {}

        # 첫 번째 ### PRODUCT_NAME을 찾고 그 이전 내용은 무시 (견고성 확보)
        # 공백 있는 버전과 없는 버전 모두 처리
        start_index = content.find("### PRODUCT_NAME")
        if start_index == -1:
            start_index = content.find("###PRODUCT_NAME")
        if start_index == -1:
            return sections

        content = content[start_index:]

        # ### [섹션 이름] \n [내용] 패턴으로 파싱 (### 뒤에 공백 허용)
        matches = re.findall(r'###\s*([A-Z_]+)\s*\n(.*?)(?=###|\Z)', content, re.DOTALL)

        for name, data in matches:
            sections[name.strip()] = data.strip()

        return sections

    def generate_keywords(
        self,
        text: str,
        max_keywords: int = 50
    ) -> KeywordResult:
        
        print(f"\n🤖 Claude 3.5 Sonnet 호출 - 키워드 생성")
        print(f"   - 텍스트 길이: {len(text)} 문자")
        print(f"   - 최대 키워드: {max_keywords}개")
        
        if not self.bedrock:
            print("⚠️ Bedrock 클라이언트 없음")
            return KeywordResult(
                product_name="N/A", type_definitions={}, combination_patterns=[],
                classified_words=[], entities_by_type={}, keywords_positive=[], keywords_negative=[],
                original_text_preview=text[:500] + "..."
            )

        # V5.X 최종 확정 프롬프트 (유연 정의 타입)
        prompt = f"""
## 🤖 NHN AD: 초정밀 엔터티 기반 광고 키워드 생성 AI 엔진

**[목표]**: 입력된 상품 텍스트를 분석하여 **광고 목적에 가장 부합하는 핵심 엔티티 타입**들을 활용해 엔티티 스키마를 구성합니다. 이 스키마를 기반으로 고전환율 및 예산 효율화에 최적화된 광고 키워드(Positive) 및 제외 키워드(Negative) 셋을 생성합니다. 출력은 토큰 효율을 위해 정해진 단순 텍스트 형식을 엄격하게 준수해야 합니다.

### 1. 입력 데이터

상품 텍스트:
{text}

### 2. 작업 1-2: 분석, 상품명 추출 및 패턴 발견

1.  **노이즈 필터링**: 가격, 할인, 약관, 리뷰, 비핵심 속성 정보 등은 원천적으로 제외합니다.
2.  **핵심 상품명 추출**: 필터링된 텍스트에서 해당 상품을 대표하는 **가장 정확하고 대표할만한 상품명**을 추출합니다.
3.  **엔티티 타입 정의**: LLM은 텍스트 맥락과 광고 목적에 가장 부합하는 **핵심 엔티티 타입**을 자유롭게 정의하고, 추출된 엔티티를 해당 타입으로 분류합니다.
4.  **핵심/속성 분류**: 자체 정의된 모든 엔티티 타입은 **핵심 엔티티 타입**과 **속성 엔티티 타입**으로 분류되어야 합니다. 추출된 **상품명**은 **핵심 엔티티 타입의 최우선 항목**으로 간주됩니다.
5.  **조합 패턴 발견**: 텍스트 맥락을 분석하여 유의미한 엔티티 조합 패턴을 발견합니다.

### 3. 작업 3-4: 키워드 생성 및 필터링

-   **규칙 준수**: 문법적 자연스러움, 특수문자 제거, 의미 중복 최소화, **4개 단어 이내 조합 제한**, 조합 순서 다양화 등 **모든 마케터 요구사항을 엄격히 적용**합니다.
-   **핵심 활용**: 추출된 **핵심 상품명**과 기타 **핵심 엔티티**는 생성되는 모든 키워드 조합에 **반드시 하나 이상** 포함되도록 최우선적으로 활용합니다.
-   **부정 키워드 생성**: 예산 낭비가 예상되는 제외 키워드를 최소 10개 이상 생성합니다.
-   **수량**: 중복 제거 후, **최대 {max_keywords} 개**의 키워드를 최종 생성합니다.

### 4. 출력 형식 (단순 텍스트 강제)

-   출력은 다음 5개 섹션으로 구성되며, 각 섹션은 **세 개의 샵(`###`) 구분자**로 시작해야 합니다.
-   **추가 지침**: 응답에는 아래 5개 섹션 외에 다른 설명, 주석, 머리말/꼬리말, 코드 블록을 포함하지 않습니다.

### PRODUCT_NAME

-   **추출된 핵심 상품명**만 단일 문자열로 출력합니다.

### DOMAIN_SCHEMA_AND_ENTITIES

-   **TYPE_DEFINITION**: (엔티티 타입1:핵심/속성); (엔티티 타입2:핵심/속성); ... 형태로 정의합니다. (엔티티 타입은 텍스트를 가장 잘 설명하도록 LLM이 정의)
-   **ENTITIES**: (엔티티 타입1): (엔터티1), (엔터티2), ... ; (엔티티 타입2): (엔터티3), ... 형태로 목록을 출력합니다.
-   **PATTERNS**: (패턴1); (패턴2); ... 형태로 조합 패턴을 세미콜론으로 구분하여 출력합니다.

### KEYWORDS_POSITIVE_TABLE

-   **5개 컬럼**을 가진 **Markdown Table** 형식으로 출력합니다.
-   컬럼 헤더는 반드시 `Keyword | User Intent | Match Type | Combination Source` 순서를 지켜야 합니다.

### KEYWORDS_NEGATIVE_LIST

-   **쉼표로 구분된** 단일 문자열로 출력합니다. (최소 10개 이상)

### END_OF_OUTPUT

-   최종 마커.
"""

        # API 호출 및 토큰 계산
        try:
            request_body = {
                "anthropic_version": "bedrock-2023-05-31",
                "messages": [
                    {"role": "user", "content": prompt}
                ],
                "max_tokens": 8192,
                "temperature": 0.2
            }

            response = self.bedrock.invoke_model(
                modelId="anthropic.claude-3-5-sonnet-20240620-v1:0",
                body=json.dumps(request_body)
            )

            response_body = json.loads(response['body'].read())
            content = response_body['content'][0]['text']

            # 원본 응답 저장 (디버깅용)
            DEBUG_OUTPUT_FILE = "claude_response_raw.txt"
            try:
                with open(DEBUG_OUTPUT_FILE, 'w', encoding='utf-8') as f:
                    f.write(content)
                print(f"📜 Claude API 원본 응답 텍스트 저장 완료: {DEBUG_OUTPUT_FILE}")
            except Exception as e:
                print(f"⚠️ 원본 응답 파일 저장 실패: {e}")

            # 토큰 및 비용 계산 로직 유지
            usage = response_body.get('usage', {})
            input_tokens = usage.get('input_tokens', 0)
            output_tokens = usage.get('output_tokens', 0)

            self.total_input_tokens += input_tokens
            self.total_output_tokens += output_tokens

            cost = self.calculate_cost(input_tokens, output_tokens)

            print(f"\n📊 토큰 사용량:")
            print(f"   - Input 토큰: {input_tokens:,}")
            print(f"   - Output 토큰: {output_tokens:,}")
            print(f"   - 총합: {input_tokens + output_tokens:,}")
            print(f"   - 비용: ${cost['total_cost_usd']:.4f} (약 {cost['total_cost_krw']:.0f}원)")
            
            # --- V5.X 출력 파싱 시작 (최종 보강 로직) ---
            
            sections = self._parse_v5_4_output(content)
            
            product_name = sections.get('PRODUCT_NAME', 'N/A').strip()

            # 2. DOMAIN_SCHEMA_AND_ENTITIES 파싱
            entities_text = sections.get('DOMAIN_SCHEMA_AND_ENTITIES', '')
            type_definitions = {}
            entities_by_type = {}
            combination_patterns = []
            classified_words = []
            keywords_positive = []
            keywords_negative = []
            
            if entities_text:
                # 1. TYPE_DEFINITION 파싱
                type_def_match = re.search(r'TYPE_DEFINITION:\s*(.*)', entities_text)
                if type_def_match:
                    type_defs_str = type_def_match.group(1).strip()
                    defs = type_defs_str.split(';')
                    for d in defs:
                        if ':' in d:
                            parts = d.split(':', 1)
                            # 괄호와 공백 제거 후 키-밸류 추출
                            key = parts[0].strip().strip('()') 
                            value = parts[1].strip().strip('()')
                            if key:
                                type_definitions[key] = value

                # 2. ENTITIES 파싱: (엔티티 타입1): 엔터티1, 엔터티2; 형태로 추출 (Final 보강)
                entities_match = re.search(r'ENTITIES:\s*(.*?)(?=PATTERNS:|\Z)', entities_text, re.DOTALL)
                if entities_match:
                    entities_str = entities_match.group(1).strip()
                    items = entities_str.split(';') # 세미콜론 기준으로 분리
                    
                    for item in items:
                        if ':' in item:
                            parts = item.split(':', 1)
                            # 괄호 제거 및 엔티티 타입 추출
                            entity_type = parts[0].strip().strip('()')
                            words_str = parts[1].strip()
                            words = [w.strip() for w in words_str.split(',') if w.strip()]
                            
                            if entity_type:
                                entities_by_type[entity_type] = words
                            
                                for word in words:
                                    classified_words.append(ClassifiedWord(
                                        word=word,
                                        entity_type=entity_type
                                    ))

                # 3. PATTERNS 파싱
                patterns_match = re.search(r'PATTERNS:\s*(.*)', entities_text)
                if patterns_match:
                    patterns_str = patterns_match.group(1).strip()
                    combination_patterns = [p.strip() for p in patterns_str.split(';') if p.strip()]

            # 4. KEYWORDS_POSITIVE_TABLE 파싱
            keywords_table_text = sections.get('KEYWORDS_POSITIVE_TABLE', '')
            if keywords_table_text:
                lines = keywords_table_text.split('\n')
                for line in lines[2:]:
                    line = line.strip()
                    if line.startswith('|') and line.endswith('|'):
                        parts = [p.strip() for p in line.split('|')[1:-1]] 
                        if parts and parts[0]:
                            keywords_positive.append(parts[0])


            # 5. KEYWORDS_NEGATIVE_LIST 파싱
            negative_list_text = sections.get('KEYWORDS_NEGATIVE_LIST', '')
            if negative_list_text:
                keywords_negative = [kw.strip() for kw in negative_list_text.split(',') if kw.strip()]
            
            # --- V5.X 출력 파싱 종료 ---
            
            # 최종 로그 출력 (수정된 파싱 로직의 성공 여부를 확인합니다)
            print(f"\n✅ 상품명 추출: {product_name}")
            print(f"✅ 엔터티 타입 정의: {len(type_definitions)}개")
            print(f"✅ 추출된 단어 총합: {len(classified_words)}개")
            print(f"✅ 조합 패턴 발견: {len(combination_patterns)}개")
            print(f"✅ 긍정 키워드 생성: {len(keywords_positive)}개")
            print(f"✅ 부정 키워드 생성: {len(keywords_negative)}개")


            return KeywordResult(
                product_name=product_name,
                type_definitions=type_definitions,
                combination_patterns=combination_patterns,
                classified_words=classified_words,
                entities_by_type=entities_by_type,
                keywords_positive=keywords_positive,
                keywords_negative=keywords_negative,
                original_text_preview=text[:500] + "..."
            )

        except Exception as e:
            print(f"⚠️ Claude 3.5 Sonnet 호출 실패: {e}")
            traceback.print_exc()
            return KeywordResult(
                product_name="N/A", type_definitions={}, combination_patterns=[],
                classified_words=[], entities_by_type={}, keywords_positive=[], keywords_negative=[],
                original_text_preview=text[:500] + "..."
            )

In [4]:
# ============================================================================
# 실행 코드
# ============================================================================
if __name__ == "__main__":
    
    TEXT_FILE_PATH = r"NOLIP_text.txt"
    OUTPUT_PATH = r"generated_keywords_v005.json"
    MAX_KEYWORDS = 100

    try:
        if not os.path.exists(TEXT_FILE_PATH):
            raise FileNotFoundError(f"텍스트 파일을 찾을 수 없습니다: {TEXT_FILE_PATH}")

        with open(TEXT_FILE_PATH, 'r', encoding='utf-8') as f:
            text = f.read()

        print(f"✅ 텍스트 읽기 완료: {len(text)} 문자")

        print("\n" + "="*80)
        print("🚀 단계 2: 키워드 생성 (Claude 3.5 Sonnet)")
        print("="*80)
        
        generator = KeywordGenerator() 
        result = generator.generate_keywords(
            text=text,
            max_keywords=MAX_KEYWORDS
        )
        
        print("\n" + "="*80)
        print("💾 단계 3: 결과 저장")
        print("="*80)
        
        output_data = {
            "product_name": result.product_name, 
            "abstract_type_definitions": result.type_definitions,
            "entities_by_type": result.entities_by_type,
            "combination_patterns": result.combination_patterns,
            "keywords_positive": result.keywords_positive, 
            "keywords_negative": result.keywords_negative, 
            "metadata": {
                "text_preview": result.original_text_preview,
                "total_defined_types": len(result.type_definitions),
                "total_entities": len(result.classified_words),
                "total_patterns": len(result.combination_patterns),
                "total_positive_keywords": len(result.keywords_positive),
                "total_negative_keywords": len(result.keywords_negative)
            }
        }

        with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
            json.dump(output_data, f, ensure_ascii=False, indent=2)

        print(f"💾 전체 결과 저장 완료: {OUTPUT_PATH}")
        print(f"   - 추출된 상품명: {result.product_name}")
        print(f"   - 정의된 엔터티 타입: {len(result.type_definitions)}개")
        print(f"   - 긍정 키워드: {len(result.keywords_positive)}개")
        print(f"   - 부정 키워드: {len(result.keywords_negative)}개")


        # =====================================================================
        # 최종 요약 (출력 강화)
        # =====================================================================
        print("\n" + "="*80)
        print("🎉 모든 단계 완료!")
        print("="*80)

        total_cost = generator.get_total_cost()

        print(f"\n📊 최종 결과:")
        print(f"   - 추출된 상품명: {result.product_name}")
        print(f"   - 추출된 조합 패턴: {len(result.combination_patterns)}개")
        print(f"   - 분류된 단어: {len(result.classified_words)}개")
        print(f"   - 생성된 긍정 키워드: {len(result.keywords_positive)}개")
        
        print("\n--- 📝 엔티티 구조 상세 (DOMAIN_SCHEMA_AND_ENTITIES) ---")
        
        # 1. 엔터티 타입 정의 (LLM이 유연하게 정의한 타입)
        print(f"**1. 정의된 엔터티 타입 ({len(result.type_definitions)}개):**")
        for etype, classification in result.type_definitions.items():
            print(f"   - {etype} ({classification})")
        
        # 2. 엔터티 타입별 단어 목록
        print(f"\n**2. 엔터티 타입별 추출된 단어:**")
        for etype, words in result.entities_by_type.items():
            # [수정] '...' 요약 제거. 항상 전체 단어 목록을 출력합니다.
            full_list = ', '.join(words)
            print(f"   - {etype} ({len(words)}개): {full_list}")

        # 3. 조합 패턴
        print(f"\n**3. 발견된 조합 패턴 ({len(result.combination_patterns)}개):**")
        for pattern in result.combination_patterns:
            print(f"   - {pattern}")
            
        # 4. 긍정 키워드 전체 목록
        # [수정] '상위 10개' -> '전체 목록'으로 변경하고 [:10] 슬라이싱 제거
        print(f"\n**4. 생성된 긍정 키워드 전체 목록 ({len(result.keywords_positive)}개):**")
        for i, kw in enumerate(result.keywords_positive, 1):
            print(f"   {i}. {kw}")
            
        # 5. 부정 키워드 전체 목록
        # [수정] '상위 10개' -> '전체 목록'으로 변경하고 '...' 요약 제거
        print(f"\n**5. 생성된 부정 키워드 전체 목록 ({len(result.keywords_negative)}개):**")
        full_neg_list = ', '.join(result.keywords_negative)
        print(f"   - {full_neg_list}")


        print(f"\n💰 총 토큰 사용량:")
        print(f"   - Input: {total_cost['input_tokens']:,} tokens")
        print(f"   - Output: {total_cost['output_tokens']:,} tokens")
        print(f"   - 총 비용: ${total_cost['total_cost_usd']:.4f} (약 {total_cost['total_cost_krw']:.0f}원)")
        print(f"\n💾 출력 파일: {OUTPUT_PATH}")

        print("\n" + "="*80)
        print("✅ 성공!")
        print("="*80)

    except FileNotFoundError as e:
        print("\n" + "="*80)
        print("❌ 파일 오류")
        print("="*80)
        print(f"\n오류: {e}")
        print("\n💡 해결 방법:")
        print("   - NOLIP_text.txt 파일이 올바른 경로에 있는지 확인하세요")

    except Exception as e:
        print("\n" + "="*80)
        print("❌ 오류 발생")
        print("="*80)
        print(f"\n오류: {e}")
        print("\n상세 정보:")
        traceback.print_exc()

✅ 텍스트 읽기 완료: 1716 문자

🚀 단계 2: 키워드 생성 (Claude 3.5 Sonnet)
✅ AWS Bedrock 클라이언트 초기화 완료

🤖 Claude 3.5 Sonnet 호출 - 키워드 생성
   - 텍스트 길이: 1716 문자
   - 최대 키워드: 100개
📜 Claude API 원본 응답 텍스트 저장 완료: claude_response_raw.txt

📊 토큰 사용량:
   - Input 토큰: 3,663
   - Output 토큰: 1,567
   - 총합: 5,230
   - 비용: $0.0345 (약 48원)

✅ 상품명 추출: 실크로드 천산북로 역사탐방 여행
✅ 엔터티 타입 정의: 7개
✅ 추출된 단어 총합: 20개
✅ 조합 패턴 발견: 5개
✅ 긍정 키워드 생성: 90개
✅ 부정 키워드 생성: 16개

💾 단계 3: 결과 저장
💾 전체 결과 저장 완료: generated_keywords_v005.json
   - 추출된 상품명: 실크로드 천산북로 역사탐방 여행
   - 정의된 엔터티 타입: 7개
   - 긍정 키워드: 90개
   - 부정 키워드: 16개

🎉 모든 단계 완료!

📊 최종 결과:
   - 추출된 상품명: 실크로드 천산북로 역사탐방 여행
   - 추출된 조합 패턴: 5개
   - 분류된 단어: 20개
   - 생성된 긍정 키워드: 90개

--- 📝 엔티티 구조 상세 (DOMAIN_SCHEMA_AND_ENTITIES) ---
**1. 정의된 엔터티 타입 (7개):**
   - PRODUCT_CATEGORY (핵심)
   - DESTINATION (핵심)
   - FEATURE (속성)
   - EXPERT (속성)
   - ACTIVITY (속성)
   - ATTRACTION (속성)
   - DURATION (속성)

**2. 엔터티 타입별 추출된 단어:**
   - PRODUCT_CATEGORY (2개): 실크로드 여행, 역사탐방
   - DESTINATION (4개): 우루무치, 투르판, 하미, 천산북로
  

In [57]:
import json
from typing import Dict, List, Optional
from dataclasses import dataclass
import os
import re
import traceback

# AWS SDK (boto3) 가용성 확인
try:
    import boto3
    BOTO3_AVAILABLE = True
except ImportError:
    BOTO3_AVAILABLE = False
    print("⚠️ boto3가 설치되지 않았습니다. Claude API 호출이 불가능합니다.")


# ============================================================================
# 데이터 클래스 (V5.Z-4)
# ============================================================================

@dataclass
class ClassifiedWord:
    """분류된 단어 (하위 엔터티 타입별)"""
    word: str
    entity_type: str

@dataclass
class PositiveKeyword:
    """'S' 플래그가 포함된 긍정 키워드"""
    keyword: str
    is_seed: bool # 'S' 플래그 (True/False)

@dataclass
class KeywordResult:
    """키워드 생성 결과"""
    product_name: str
    combination_patterns: List[str]
    # [수정] List<> -> List[]로 변경
    classified_words: List[ClassifiedWord] 
    entities_by_type: Dict[str, List[str]]
    keywords_positive: List[PositiveKeyword]
    original_text_preview: str


# ============================================================================
# 키워드 생성기 클래스 (V5.Z-7: Concise Heuristic)
# ============================================================================

class KeywordGenerator:
    """
    V5.Z-7 (최종) 'S' (시드 키워드) 플래그 기능을 토큰 효율적으로 추가
    """

    def __init__(self, region: str = "ap-northeast-2"):
        self.region = region
        self.total_input_tokens = 0
        self.total_output_tokens = 0

        if BOTO3_AVAILABLE:
            try:
                self.bedrock = boto3.client(
                    service_name='bedrock-runtime',
                    region_name=region,
                    config=boto3.session.Config(
                        read_timeout=300,
                        retries={'max_attempts': 3}
                    )
                )
                print("✅ AWS Bedrock 클라이언트 초기화 완료")
            except Exception as e:
                print(f"⚠️ AWS Bedrock 초기화 실패: {e}")
                self.bedrock = None
        else:
            self.bedrock = None

    def calculate_cost(self, input_tokens: int, output_tokens: int) -> dict:
        """토큰 사용량 기반 비용 계산 (Claude 3.5 Sonnet 가격 기준)"""
        INPUT_PRICE_PER_1M = 3.00
        OUTPUT_PRICE_PER_1M = 15.00
        USD_TO_KRW = 1400

        input_cost_usd = (input_tokens / 1000000) * INPUT_PRICE_PER_1M
        output_cost_usd = (output_tokens / 1000000) * OUTPUT_PRICE_PER_1M
        total_cost_usd = input_cost_usd + output_cost_usd
        total_cost_krw = total_cost_usd * USD_TO_KRW

        return {
            'input_tokens': input_tokens,
            'output_tokens': output_tokens,
            'input_cost_usd': input_cost_usd,
            'output_cost_usd': output_cost_usd,
            'total_cost_usd': total_cost_usd,
            'total_cost_krw': total_cost_krw
        }

    def get_total_cost(self) -> dict:
        return self.calculate_cost(self.total_input_tokens, self.total_output_tokens)

    def _parse_v5_4_output(self, content: str) -> Dict[str, str]:
        """### 구분자 기반 출력을 파싱 (최종 보강)"""
        sections = {}

        start_index = content.find("### PRODUCT_NAME")
        if start_index == -1:
            start_index = content.find("###PRODUCT_NAME")
        if start_index == -1:
            return sections

        content = content[start_index:]
        matches = re.findall(r'###\s*([A-Z_]+)\s*\n(.*?)(?=###|\Z)', content, re.DOTALL)

        for name, data in matches:
            sections[name.strip()] = data.strip()

        return sections

    def generate_keywords(
        self,
        text: str,
        max_keywords: int = 50
    ) -> KeywordResult:
        
        print(f"\n🤖 Claude 3.5 Sonnet 호출 - 키워드 생성")
        print(f"   - 텍스트 길이: {len(text)} 문자")
        print(f"   - 최대 키워드: {max_keywords}개")
        
        if not self.bedrock:
            print("⚠️ Bedrock 클라이언트 없음")
            return KeywordResult(
                product_name="N/A", combination_patterns=[],
                classified_words=[], entities_by_type={}, keywords_positive=[],
                original_text_preview=text[:500] + "..."
            )

        # ====================================================================
        # [변경점 1] 프롬프트 수정 (V5.Z-7)
        # - 작업 2 (상품명 추출)를 간결한 단일 문장으로 수정
        # ====================================================================
        prompt = f"""
## 🤖 NHN AD: 초정밀 엔터티 기반 광고 키워드 생성 AI 엔진 (V5.Z-7: Concise Heuristic)

**[목표]**: 입력된 상품 텍스트를 분석하여 광고 목적에 가장 부합하는 **핵심 엔티티 타입**과 **속성 엔티티 타입**으로 엔티티 스키마를 구성합니다. 이 스키마를 기반으로 고전환율에 최적화된 광고 키워드(Positive) 셋을 생성합니다.

### 1. 입력 데이터

상품 텍스트:
{text}

### 2. 작업: 엔티티 스키마 구성 및 키워드 생성

1.  **분석 및 필터링**: 입력 텍스트를 분석합니다. 가격, 할인, 약관, 리뷰 등 광고 키워드와 관련 없는 노이즈 정보는 제외합니다.
2.  **핵심 상품명 추출 (Internal 2-Step CoT)**:
    * **(Step 1: 내부 추론)** 먼저 텍스트 전체의 맥락(주소, 리뷰, 관련 설명 등)을 살펴 **'주제(Subject)'가 되는 상품**을 **마음속으로(internally) 추론**합니다.
    * **(Step 2: 원본 추출)** 다음으로, 1단계에서 추론한 주제와 일치하는 **'정확한 원본 단일 라인 제목'**을 텍스트에서 **찾아냅니다.**
    * 이 **'추출된 원본 제목'**을 `### PRODUCT_NAME` 섹션에 출력합니다.
3.  **유연한 엔티티 타입 정의**: 텍스트의 맥락과 광고 목적에 맞게 엔티티 타입을 **자유롭게 정의**합니다. (예: `PRODUCT_CATEGORY`, `DESTINATION`)
4.  **(내부 사고 1)**: 3단계에서 정의한 모든 타입을 '핵심' 또는 '속성'으로 **마음속으로(internally) 분류**합니다.
5.  **[규칙]**: 4단계의 분류 작업(핵심/속성)은 **최종 출력에 절대 포함하지 않습니다.** 하지만 8번 작업(키워드 생성) 시 '핵심 엔티티 포함' 규칙을 지키기 위해 **반드시 수행**해야 합니다.
6.  **엔티티 추출 및 매핑**: 텍스트에서 광고 키워드 조합에 유의미한 엔티티 단어들을 **빠짐없이** 추출하여 3단계에서 정의한 타입에 매핑합니다. (출력 섹션: `### DOMAIN_SCHEMA_AND_ENTITIES`의 `ENTITIES`)
7.  **조합 패턴 발견**: 텍스트 맥락에서 유의미한 엔티티 조합 패턴을 발견합니다. (예: `PRODUCT_CATEGORY + DESTINATION`) (출력 섹션: `### DOMAIN_SCHEMA_AND_ENTITIES`의 `PATTERNS`)
8.  **키워드 생성 (Positive)**:
    * **핵심 우선**: **(내부적으로 분류한) '핵심' 엔티티**가 **반드시 1개 이상 포함**되도록 조합합니다.
    * **제한**: 문법적으로 자연스럽고, 특수문자를 제거하며, 의미 중복을 최소화하고, **최대 4개 단어**로 조합합니다.
    * **수량**: 중복 제거 후 **최대 {max_keywords} 개**의 긍정 키워드를 생성합니다. (출력 섹션: `### KEYWORDS_POSITIVE_TABLE`)
9.  **시드(Seed) 키워드 식별 (S)**: 8단계에서 생성된 긍정 키워드에, 2단계에서 추출된 **핵심 상품명** 의 단어 중 **하나라도 포함(CONTAINS)되어 있으면** 해당 키워드를 **'시드 키워드(S)'**로 식별합니다.
10. **(내부 사고 2)**: 예산 낭비가 예상되는 제외 키워드(Negative)를 **마음속으로(internally) 최소 10개 이상** 생각합니다. (예: 호텔, 항공권, 패키지, 할인...)
11. **[규칙]**: 10단계에서 생성한 제외 키워드 목록은 토큰 절약을 위해 **최종 출력에 절대 포함하지 않습니다.**

### 3. 출력 형식 (엄격한 규칙)

-   출력은 아래 **3개** 섹션으로 구성되며, 각 섹션은 **세 개의 샵(`###`) 구분자**로 시작해야 합니다.
-   **[절대 규칙]**: 아래 3개 섹션 외에 **그 어떤 설명, 주석, 머리말, 꼬리말, 코드 블록(` ``` `)도 절대 포함하지 마십시오.** 응답은 `### PRODUCT_NAME`으로 즉시 시작해야 합니다.

### PRODUCT_NAME

-   **추출된 핵심 상품명**만 단일 문자열로 출력합니다.

### DOMAIN_SCHEMA_AND_ENTITIES

-   **ENTITIES**: 엔티티 타입1: 엔터티1, 엔터티2; 엔티티 타입2: 엔터티3; ... 형태로 목록을 출력합니다. (괄호 없음)
-   **PATTERNS**: 패턴1; 패턴2; ... 형태로 조합 패턴을 세미콜론으로 구분하여 출력합니다. (괄호 없음)
-   **[형식 예시]**
    ENTITIES: BRAND: 나이키, 아디다스; PRODUCT_TYPE: 운동화, 런닝화
    PATTERNS: BRAND + PRODUCT_TYPE; BRAND + PRODUCT_TYPE + COLOR

### KEYWORDS_POSITIVE_TABLE

-   긍정 키워드를 **줄바꿈(newline)으로 구분**하여 단순 텍스트 목록으로 출력합니다.
-   **[시드(S) 표기]**: 9단계에서 '시드 키워드'로 식별된 경우, 키워드 끝에 `|S`를 추가로 붙입니다. (토큰 절약형)
-   **[규칙]**: 시드 키워드가 아닌 경우, `|S`를 붙이지 않습니다. (예: `일반키워드`)
-   [절대 규칙]: 마크다운 테이블, 쉼표, `|S` 외의 다른 `|` 문자, 헤더 등을 절대 포함하지 마십시오.
-   [형식 예시]
    긍정키워드1|S
    긍정키워드2
    긍정키워드3|S
    긍정키워드4

### END_OF_OUTPUT

-   최종 마커.
"""

        # API 호출 및 토큰 계산
        try:
            # ... (이하 API 호출, 파싱, return 로직은 V5.Z-5/6와 모두 동일) ...
            
            request_body = {
                "anthropic_version": "bedrock-2023-05-31",
                "messages": [
                    {"role": "user", "content": prompt}
                ],
                "max_tokens": 8192,
                "temperature": 0.1
            }

            response = self.bedrock.invoke_model(
                modelId="anthropic.claude-3-5-sonnet-20240620-v1:0",
                body=json.dumps(request_body)
            )

            response_body = json.loads(response['body'].read())
            content = response_body['content'][0]['text']

            DEBUG_OUTPUT_FILE = "claude_response_raw.txt"
            try:
                with open(DEBUG_OUTPUT_FILE, 'w', encoding='utf-8') as f:
                    f.write(content)
                print(f"📜 Claude API 원본 응답 텍스트 저장 완료: {DEBUG_OUTPUT_FILE}")
            except Exception as e:
                print(f"⚠️ 원본 응답 파일 저장 실패: {e}")

            usage = response_body.get('usage', {})
            input_tokens = usage.get('input_tokens', 0)
            output_tokens = usage.get('output_tokens', 0)

            self.total_input_tokens += input_tokens
            self.total_output_tokens += output_tokens

            cost = self.calculate_cost(input_tokens, output_tokens)

            print(f"\n📊 토큰 사용량:")
            print(f"   - Input 토큰: {input_tokens:,}")
            print(f"   - Output 토큰: {output_tokens:,}")
            print(f"   - 총합: {input_tokens + output_tokens:,}")
            print(f"   - 비용: ${cost['total_cost_usd']:.4f} (약 {cost['total_cost_krw']:.0f}원)")
            
            sections = self._parse_v5_4_output(content)
            
            product_name = sections.get('PRODUCT_NAME', 'N/A').strip()

            entities_text = sections.get('DOMAIN_SCHEMA_AND_ENTITIES', '')
            
            entities_by_type = {}
            combination_patterns = []
            classified_words = []
            keywords_positive: List[PositiveKeyword] = []
            
            if entities_text:
                entities_match = re.search(r'ENTITIES:\s*(.*?)(?=PATTERNS:|\Z)', entities_text, re.DOTALL)
                if entities_match:
                    entities_str = entities_match.group(1).strip()
                    items = entities_str.split(';') 
                    
                    for item in items:
                        if ':' in item:
                            parts = item.split(':', 1)
                            entity_type = parts[0].strip()
                            words_str = parts[1].strip()
                            words = [w.strip() for w in words_str.split(',') if w.strip()]
                            
                            if entity_type:
                                entities_by_type[entity_type] = words
                            
                                for word in words:
                                    classified_words.append(ClassifiedWord(
                                        word=word,
                                        entity_type=entity_type
                                    ))

                patterns_match = re.search(r'PATTERNS:\s*(.*)', entities_text)
                if patterns_match:
                    patterns_str = patterns_match.group(1).strip()
                    patterns_list = [p.strip().strip('()') for p in patterns_str.split(';') if p.strip()]
                    combination_patterns = [p for p in patterns_list if p] 

            keywords_table_text = sections.get('KEYWORDS_POSITIVE_TABLE', '')
            if keywords_table_text:
                lines = keywords_table_text.split('\n')
                for line in lines:
                    line = line.strip()
                    if not line:
                        continue
                    
                    is_seed = False
                    kw = line
                    
                    if line.endswith('|S'):
                        kw = line[:-2].strip() 
                        is_seed = True
                    
                    if kw: 
                        keywords_positive.append(PositiveKeyword(
                            keyword=kw,
                            is_seed=is_seed
                        ))
            
            print(f"\n✅ 상품명 추출: {product_name}")
            print(f"✅ 추출된 단어 총합: {len(classified_words)}개")
            print(f"✅ 조합 패턴 발견: {len(combination_patterns)}개")
            print(f"✅ 긍정 키워드 생성: {len(keywords_positive)}개")


            return KeywordResult(
                product_name=product_name,
                combination_patterns=combination_patterns,
                classified_words=classified_words,
                entities_by_type=entities_by_type,
                keywords_positive=keywords_positive,
                original_text_preview=text[:500] + "..."
            )

        except Exception as e:
            print(f"⚠️ Claude 3.5 Sonnet 호출 실패: {e}")
            traceback.print_exc()
            return KeywordResult(
                product_name="N/A", combination_patterns=[],
                classified_words=[], entities_by_type={}, keywords_positive=[],
                original_text_preview=text[:500] + "..."
            )

In [63]:
# ============================================================================
# 실행 코드
# ============================================================================
if __name__ == "__main__":
    
    TEXT_FILE_PATH = r"NOLIP_text.txt"
    OUTPUT_PATH = r"generated_keywords_v005.json"
    MAX_KEYWORDS = 100

    try:
        if not os.path.exists(TEXT_FILE_PATH):
            raise FileNotFoundError(f"텍스트 파일을 찾을 수 없습니다: {TEXT_FILE_PATH}")

        with open(TEXT_FILE_PATH, 'r', encoding='utf-8') as f:
            text = f.read()

        print(f"✅ 텍스트 읽기 완료: {len(text)} 문자")

        print("\n" + "="*80)
        print("🚀 단계 2: 키워드 생성 (Claude 3.5 Sonnet)")
        print("="*80)
        
        generator = KeywordGenerator() 
        result = generator.generate_keywords(
            text=text,
            max_keywords=MAX_KEYWORDS
        )
        
        print("\n" + "="*80)
        print("💾 단계 3: 결과 저장")
        print("="*80)
        
        # ====================================================================
        # [오류 수정] JSON 직렬화를 위해 List[PositiveKeyword] -> List[Dict] 변환
        # ====================================================================
        positive_keywords_for_json = [
            {"keyword": kw.keyword, "is_seed": kw.is_seed}
            for kw in result.keywords_positive
        ]
        
        output_data = {
            "product_name": result.product_name, 
            "entities_by_type": result.entities_by_type,
            "combination_patterns": result.combination_patterns,
            "keywords_positive": positive_keywords_for_json, # <--- 변환된 리스트를 사용
            "metadata": {
                "text_preview": result.original_text_preview,
                "total_entities": len(result.classified_words),
                "total_patterns": len(result.combination_patterns),
                "total_positive_keywords": len(result.keywords_positive),
            }
        }

        with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
            json.dump(output_data, f, ensure_ascii=False, indent=2)

        print(f"💾 전체 결과 저장 완료: {OUTPUT_PATH}")
        print(f"   - 추출된 상품명: {result.product_name}")
        print(f"   - 긍정 키워드: {len(result.keywords_positive)}개")
        # (부정 키워드 로그는 이미 제거됨)


        # =====================================================================
        # 최종 요약 (출력 강화)
        # =====================================================================
        print("\n" + "="*80)
        print("🎉 모든 단계 완료!")
        print("="*80)

        total_cost = generator.get_total_cost()

        print(f"\n📊 최종 결과:")
        print(f"   - 추출된 상품명: {result.product_name}")
        print(f"   - 추출된 조합 패턴: {len(result.combination_patterns)}개")
        print(f"   - 분류된 단어: {len(result.classified_words)}개")
        print(f"   - 생성된 긍정 키워드: {len(result.keywords_positive)}개")
        
        print("\n--- 📝 엔티티 구조 상세 (DOMAIN_SCHEMA_AND_ENTITIES) ---")
        
        # 1. 엔터티 타입별 단어 목록 (번호 1번 유지)
        print(f"\n**1. 엔터티 타입별 추출된 단어:**")
        for etype, words in result.entities_by_type.items():
            full_list = ', '.join(words)
            print(f"   - {etype} ({len(words)}개): {full_list}")

        # 2. 발견된 조합 패턴 (번호 2번 유지)
        print(f"\n**2. 발견된 조합 패턴 ({len(result.combination_patterns)}개):**")
        for pattern in result.combination_patterns:
            print(f"   - {pattern}")
            
        # 3. 긍정 키워드 전체 목록 (출력 시 [S] 플래그 표시)
        print(f"\n**3. 생성된 긍정 키워드 전체 목록 ({len(result.keywords_positive)}개):**")
        for i, kw_obj in enumerate(result.keywords_positive, 1):
            seed_flag = " [S]" if kw_obj.is_seed else "" # 'S' 플래그가 True면 [S] 표시
            print(f"   {i}. {kw_obj.keyword}{seed_flag}")
            
        # (부정 키워드 섹션은 이미 제거됨)


        print(f"\n💰 총 토큰 사용량:")
        print(f"   - Input: {total_cost['input_tokens']:,} tokens")
        print(f"   - Output: {total_cost['output_tokens']:,} tokens")
        print(f"   - 총 비용: ${total_cost['total_cost_usd']:.4f} (약 {total_cost['total_cost_krw']:.0f}원)")
        print(f"\n💾 출력 파일: {OUTPUT_PATH}")

        print("\n" + "="*80)
        print("✅ 성공!")
        print("="*80)

    except FileNotFoundError as e:
        # ... (오류 처리 동일) ...
        print("\n" + "="*80)
        print("❌ 파일 오류")
        print("="*80)
        print(f"\n오류: {e}")
        print("\n💡 해결 방법:")
        print("   - NOLIP_text.txt 파일이 올바른 경로에 있는지 확인하세요")

    except Exception as e:
        # ... (오류 처리 동일) ...
        print("\n" + "="*80)
        print("❌ 오류 발생")
        print("="*80)
        print(f"\n오류: {e}")
        print("\n상세 정보:")
        traceback.print_exc()

✅ 텍스트 읽기 완료: 2535 문자

🚀 단계 2: 키워드 생성 (Claude 3.5 Sonnet)
✅ AWS Bedrock 클라이언트 초기화 완료

🤖 Claude 3.5 Sonnet 호출 - 키워드 생성
   - 텍스트 길이: 2535 문자
   - 최대 키워드: 100개
📜 Claude API 원본 응답 텍스트 저장 완료: claude_response_raw.txt

📊 토큰 사용량:
   - Input 토큰: 4,911
   - Output 토큰: 1,427
   - 총합: 6,338
   - 비용: $0.0361 (약 51원)

✅ 상품명 추출: 보마 리조트 나트랑
✅ 추출된 단어 총합: 20개
✅ 조합 패턴 발견: 5개
✅ 긍정 키워드 생성: 94개

💾 단계 3: 결과 저장
💾 전체 결과 저장 완료: generated_keywords_v005.json
   - 추출된 상품명: 보마 리조트 나트랑
   - 긍정 키워드: 94개

🎉 모든 단계 완료!

📊 최종 결과:
   - 추출된 상품명: 보마 리조트 나트랑
   - 추출된 조합 패턴: 5개
   - 분류된 단어: 20개
   - 생성된 긍정 키워드: 94개

--- 📝 엔티티 구조 상세 (DOMAIN_SCHEMA_AND_ENTITIES) ---

**1. 엔터티 타입별 추출된 단어:**
   - RESORT (1개): 보마 리조트
   - LOCATION (2개): 나트랑, 베트남
   - AMENITIES (5개): 수영장, 스파, 피트니스 센터, 레스토랑, 바
   - SERVICES (4개): 공항 교통편, 컨시어지, 룸서비스, 세탁 서비스
   - ROOM_FEATURES (4개): 에어컨, 미니바, 스마트 TV, 무료 Wi-Fi
   - NEARBY_ATTRACTIONS (4개): 혼 총, 혼총 곶, 바이 두옹 비치, 포나가르 사원

**2. 발견된 조합 패턴 (5개):**
   - RESORT + LOCATION
   - LOCATION + AMENITIES
   - RESORT +